In [1]:
import polars as pl
import pandas as pd



In [ ]:
tracking_df = pd.read_hdf("tracking_results/DeepEcoHAB_20260501_211615_007DLC_DekrW18_the_future_of_EcohabNov22shuffle1_snapshot_best-260_el.h5")

long_pdf = (
    tracking_df
    .stack(["scorer", "individuals", "bodyparts", "coords"])
    .rename("value")
    .reset_index()
)
tracking_lf = pl.from_pandas(long_pdf).rename({'level_0':frame}).drop('scorer') #ograć rename wczytywaniem z hdf

/tmp/ipykernel_90434/3250080919.py:5: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(["scorer", "individuals", "bodyparts", "coords"])


In [ ]:
antenna_df = pl.scan_csv(
    "tracking_results/DeepEcoHAB_20260501_211615_antenna_data.csv", 
    has_header=False,
    separator=";",
    new_columns=['antenna', 'tag', 'timestamp', 'datetime'],
    infer_schema_length=1000
    ).with_columns(
        pl.col('timestamp').str.to_integer(base=16)
    )

video_index = 7 #to extract from filename suffix
frames_per_video = 108000

antenna_df = antenna_df.with_columns(
    (pl.col('antenna')==10).cum_sum().alias('occurence')
).filter(
    pl.col('occurence').is_between(frames_per_video*video_index, frames_per_video*(video_index+1))
)